# S22 · The SVM — the widest street

A bank wants one clean line that separates **safe** loan applicants from **risky**
ones, so a new application lands cleanly on one side. Many lines could split two
groups. The **support vector machine (SVM)** picks a special one: the line with the
widest empty "street" on either side, because that leaves the most room to be a
little wrong on a new case.

In the main path we let scikit-learn's ready-made SVM do the work on a small,
labelled example, then we *read* its answer: the dividing line, the widest street,
and the handful of applicants that hold the line in place. No formulas needed. The
optional stretch cell at the end builds the same SVM by hand in CVXPY for anyone
curious how it works underneath.

**New here? Read this once.**

- New to Python? You can still run every cell with the play button and read the
  plain-English note above each one. The main path uses a ready-made tool, so there
  is no maths to write.
- New to the "bowl vs wobbly" idea? Open `primers/convex_sets_and_functions.md`.
- Already confident? Jump to the **Stretch (optional)** cell near the end, where we
  build the SVM from scratch as a bowl-with-fences in CVXPY.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. It installs CVXPY, which only the
optional stretch cell needs. On your **own machine** you installed everything with
`uv`, so the cell does nothing there.

In [ ]:
# CVXPY is only needed for the optional stretch cell. Install it on Colab just in case.
import sys
if "google.colab" in sys.modules:
    !pip install -q cvxpy
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — import the tools we need

NumPy for arrays, Matplotlib for the picture, and scikit-learn's ready-made SVM,
`SVC`. That is all the main path needs.

In [ ]:
import numpy as np                 # arrays and maths
import matplotlib.pyplot as plt    # plots
from sklearn.svm import SVC        # scikit-learn's ready-made support vector machine

# Set a seed so the data is the same every run.
np.random.seed(3)

print("Tools ready.")

## Step 2 — make a small set of loan applicants

Each applicant has two numbers a bank looks at: **monthly income** (in Rs thousands)
and **existing EMIs** — the Rs thousands they already pay every month on other loans.
Safe applicants tend to earn more and owe less; risky ones earn less and owe more.

We invent 40 applicants, 20 safe and 20 risky, with seeded random numbers so the set
is the same every run. We label safe applicants `+1` and risky ones `-1`, because
that is the convention the SVM uses.

In [ ]:
number_each = 20

# Safe applicants: higher income, lower existing EMIs.
safe_income = np.random.normal(75, 6, number_each)
safe_emis   = np.random.normal(12, 3, number_each)

# Risky applicants: lower income, higher existing EMIs.
risky_income = np.random.normal(45, 6, number_each)
risky_emis   = np.random.normal(28, 3, number_each)

# Stack them into one table: column 0 is income, column 1 is existing_emis.
income        = np.concatenate([safe_income, risky_income])
existing_emis = np.concatenate([safe_emis, risky_emis])
applicants = np.column_stack([income, existing_emis])

# Labels: +1 for the 20 safe applicants, -1 for the 20 risky ones.
labels = np.concatenate([np.ones(number_each), -np.ones(number_each)])

print("number of applicants:", applicants.shape[0])
print("first 3 safe   [income, emis]:", applicants[:3].round(1).tolist())
print("first 3 risky  [income, emis]:", applicants[20:23].round(1).tolist())

## Step 3 — look at the data first

Always plot before modelling. Safe applicants (blue) should sit up-and-left (more
income, fewer EMIs); risky ones (red) down-and-right. If the two clouds are
separate, a straight line can divide them.

In [ ]:
safe = applicants[labels == 1]
risky = applicants[labels == -1]

plt.figure(figsize=(7, 6))
plt.scatter(safe[:, 0], safe[:, 1], color="#2E75B6", s=50, label="safe (+1)")
plt.scatter(risky[:, 0], risky[:, 1], color="#C0392B", s=50, marker="s", label="risky (-1)")
plt.xlabel("monthly income (Rs thousands)")
plt.ylabel("existing EMIs (Rs thousands)")
plt.title("Loan applicants: safe vs risky")
plt.legend()
plt.show()

## Step 4 — let scikit-learn draw the widest street

We hand the applicants and their labels to `SVC`, scikit-learn's support vector
machine, and ask for a straight (`linear`) dividing line. The setting `C=1000` just
tells it to insist the two groups stay cleanly apart. One `.fit(...)` call and it has
found the widest-street line for us.

In [ ]:
loan_svm = SVC(kernel="linear", C=1000.0)
loan_svm.fit(applicants, labels)

# How well does the line separate the two groups on this data?
accuracy = loan_svm.score(applicants, labels)
print("share of applicants placed on the correct side:", accuracy)

## Step 5 — draw the line, the street, and the applicants that fix it

scikit-learn gives us everything we need to picture the answer, no formulas required:

- the **solid line** is the bank's dividing boundary,
- the two **dashed lines** are the edges of the widest empty "street",
- the **circled points** are the *support vectors*: the few applicants sitting right
  on the street's edge, who alone pin the line in place.

We use the model's own `decision_function` to draw the boundary and street, and its
`support_vectors_` to circle the points that matter.

In [ ]:
plt.figure(figsize=(8, 7))

# The applicants, coloured by group.
plt.scatter(safe[:, 0], safe[:, 1], color="#2E75B6", s=50, label="safe (+1)")
plt.scatter(risky[:, 0], risky[:, 1], color="#C0392B", s=50, marker="s", label="risky (-1)")

# Circle the support vectors that the model itself identified.
support_points = loan_svm.support_vectors_
plt.scatter(support_points[:, 0], support_points[:, 1],
            facecolors="none", edgecolors="#1E8449", s=260, linewidths=2.5,
            label="support vectors")

# Ask the model to score a grid of points, then let Matplotlib trace the lines
# where the score is -1, 0 and +1. Score 0 is the boundary; -1 and +1 are the edges.
x_min, x_max = income.min() - 3, income.max() + 3
y_min, y_max = existing_emis.min() - 3, existing_emis.max() + 3
grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])
scores = loan_svm.decision_function(grid_points).reshape(grid_x.shape)

plt.contour(grid_x, grid_y, scores, levels=[0], colors="#1E8449", linewidths=2.5)
plt.contour(grid_x, grid_y, scores, levels=[-1, 1], colors="#7F8C8D",
            linewidths=1.5, linestyles="--")

plt.xlabel("monthly income (Rs thousands)")
plt.ylabel("existing EMIs (Rs thousands)")
plt.title("Widest-street SVM: the boundary, the street, and the support vectors")
plt.legend(loc="best")
plt.show()

## Step 6 — read off the applicants that hold the line

The whole boundary is decided by just a handful of applicants, the support vectors,
sitting on the edge of the street. Every other applicant could shift a little and the
line would not move at all. Let us see how few there are and who they are.

In [ ]:
print("number of support vectors (applicants that fix the line):",
      loan_svm.support_vectors_.shape[0])
print()
print("their [income, existing_emis], in Rs thousands:")
for point in loan_svm.support_vectors_:
    print(f"   income {point[0]:5.1f}   emis {point[1]:5.1f}")
print()
print("A few applicants on the edge decide the bank's cutoff for everyone else.")

## What you just did (this is the main point)

You took a labelled business example, safe vs risky loan applicants, and let a
ready-made SVM find the dividing line with the **widest street** between the groups.
You read its answer straight off the picture: the boundary, the street, and the few
**support vectors** that hold it in place. No formula, just interpretation.

And you can name why this works: the widest-street problem is another **bowl-shaped
(convex)** problem, so the solver returns the single best line every time. It is the
same family as the budget split and the portfolio — describe the goal and the rules,
and a solver finds the best. The stretch cell below builds this SVM by hand for
anyone who wants to see the machinery; you do not need it to have done real work.

### Stretch (optional) — build the same SVM by hand, as a bowl-with-fences

Skip this if you are new. Here is what scikit-learn did for us underneath. An SVM
looks for a line `w·x + b = 0` and asks for two things at once:

- make `½‖w‖²` as small as possible — the trick that *widens* the street, and
- keep every point on its correct side by a full margin: `label·(w·x + b) ≥ 1` for
  every applicant.

The first part is a bowl; the second is one straight-line **fence** per applicant. A
bowl with straight-line fences is a **convex quadratic program (QP)**, so CVXPY
solves it to the single best answer. We write it out and check it matches
scikit-learn.

In [ ]:
import cvxpy as cp

# The unknowns: the line's direction w (2 numbers) and its offset b.
w = cp.Variable(2)
b = cp.Variable()

# Goal: half the squared length of w (this is what widens the street).
objective = cp.Minimize(0.5 * cp.sum_squares(w))

# One fence per applicant: it must be on its correct side by at least 1.
fences = []
for i in range(applicants.shape[0]):
    fences.append(labels[i] * (w @ applicants[i] + b) >= 1)

problem = cp.Problem(objective, fences)
problem.solve()

print("Solver status:", problem.status)
print("our w (direction):", w.value.round(4), "   sklearn w:", loan_svm.coef_[0].round(4))
print("our b (offset)   :", round(float(b.value), 4),
      "   sklearn b:", round(float(loan_svm.intercept_[0]), 4))
print()
print("Same line as scikit-learn found - we built a real SVM in CVXPY.")

The **support vectors** are the applicants whose fence is tight, where
`label·(w·x + b)` equals exactly 1. Those, and only those, hold the line. We pick
them out and confirm they are the same points scikit-learn circled.

In [ ]:
margin_values = labels * (applicants @ w.value + b.value)

support_vector_indexes = []
for i in range(applicants.shape[0]):
    if abs(margin_values[i] - 1) < 0.01:      # sitting right on the street's edge
        support_vector_indexes.append(i)

print("our support vectors    :", support_vector_indexes)
print("their [income, emis]   :", applicants[support_vector_indexes].round(1).tolist())
print()
print("count from CVXPY   :", len(support_vector_indexes))
print("count from sklearn :", loan_svm.support_vectors_.shape[0])

Finally, the **boundary algebra** the main path avoided. The line `w·x + b = 0` can
be rearranged for the vertical axis, `emis = -(w0·income + b) / w1`, and the street
edges are the same with `b` replaced by `b − 1` and `b + 1`. We draw them explicitly
here so you can see the geometry the `decision_function` hid.

In [ ]:
income_grid = np.linspace(income.min() - 3, income.max() + 3, 50)

boundary = -(w.value[0] * income_grid + b.value) / w.value[1]
upper    = -(w.value[0] * income_grid + b.value - 1) / w.value[1]
lower    = -(w.value[0] * income_grid + b.value + 1) / w.value[1]

plt.figure(figsize=(8, 7))
plt.scatter(safe[:, 0], safe[:, 1], color="#2E75B6", s=50, label="safe (+1)")
plt.scatter(risky[:, 0], risky[:, 1], color="#C0392B", s=50, marker="s", label="risky (-1)")
plt.scatter(applicants[support_vector_indexes][:, 0],
            applicants[support_vector_indexes][:, 1],
            facecolors="none", edgecolors="#1E8449", s=260, linewidths=2.5,
            label="support vectors")
plt.plot(income_grid, boundary, color="#1E8449", linewidth=2.5, label="boundary")
plt.plot(income_grid, upper, color="#7F8C8D", linewidth=1.5, linestyle="--")
plt.plot(income_grid, lower, color="#7F8C8D", linewidth=1.5, linestyle="--")
plt.xlabel("monthly income (Rs thousands)")
plt.ylabel("existing EMIs (Rs thousands)")
plt.title("The same SVM, drawn from the CVXPY line by hand")
plt.legend(loc="best")
plt.show()

## What you just did

Underneath scikit-learn's one-line SVM sits a **bowl-with-fences**: minimise `½‖w‖²`
(to widen the street) subject to one straight-line fence per applicant
(`label·(w·x + b) ≥ 1`). You wrote it out in CVXPY, got the same line scikit-learn
did, and picked out the same support vectors.

The theme of the whole session holds once more: a huge family of real problems are
bowl-shaped (convex), so you can describe the goal and the rules and let a solver
hand back the guaranteed best answer.